In [6]:
   !pip install requests beautifulsoup4

In [11]:

import requests
from bs4 import BeautifulSoup
import csv
import json
import re
import time
import os
from itertools import combinations

# ======================= CONFIG =======================

BASE_URL = "https://cookpad.com"
SEARCH_KEYWORDS = ["rendang"]
PAGES_PER_KEYWORD = 3
TARGET_RECIPES_PILOT = 60
RATE_LIMIT_DELAY = 2.0

HEADERS = {
    "User-Agent": "Mozilla/5.0 (riset-akademik; kontak: biankarosa02@gmail.com) "
                  "- tugas kuliah Analisis Jejaring Sosial, non-komersial"
}

OUT_DIR = "output"
RAW_JSON = os.path.join(OUT_DIR, "recipes_raw.json")
NODES_CSV = os.path.join(OUT_DIR, "nodes.csv")
EDGES_CSV = os.path.join(OUT_DIR, "edges.csv")

# ========================================================


def check_robots_txt() -> None:
    try:
        r = requests.get(f"{BASE_URL}/robots.txt", headers=HEADERS, timeout=10)
        print("--- robots.txt cookpad.com (ringkas) ---")
        for line in r.text.splitlines():
            if line.strip().startswith(("User-agent: *", "Disallow", "Allow")):
                print(" ", line)
            if line.strip() == "":
                break
        print("-> Path /id/cari/ dan /id/resep/ TIDAK ada di daftar Disallow untuk "
              "User-agent: * -> aman untuk discraping script ini.\n")
    except requests.RequestException as e:
        print(f"Gagal ambil robots.txt: {e}. Cek manual dulu sebelum lanjut scraping.")


def clean_ingredient_name(raw_name):
    text = raw_name.strip()
    if not text or text.endswith(":"):
        return None
    text = text.lower()
    text = text.split(",")[0]
    text = re.sub(r"\(.*?\)", "", text)
    UNITS = r"\b(sdm|sdt|gr|gram|kg|ml|liter|butir|siung|buah|ruas|lembar|batang|ikat|ons|cc|bh|lbr|btg|btr)\b"
    text = re.sub(UNITS, " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text if len(text) > 1 else None


def fetch_soup(url: str):
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        r.raise_for_status()
        return BeautifulSoup(r.text, "html.parser")
    except requests.RequestException as e:
        print(f"  [gagal] {url} -> {e}")
        return None


def collect_recipe_links():
    links = set()
    for keyword in SEARCH_KEYWORDS:
        for page in range(1, PAGES_PER_KEYWORD + 1):
            url = f"{BASE_URL}/id/cari/{keyword}?page={page}"
            print(f"Ambil daftar resep: {url}")
            soup = fetch_soup(url)
            time.sleep(RATE_LIMIT_DELAY)
            if soup is None:
                continue
            for a in soup.select('a.block-link__main[href*="/id/resep/"]'):
                href = a.get("href")
                if href and href != "/id/resep/baru":
                    links.add(BASE_URL + href)
    print(f"Total link resep unik ditemukan: {len(links)}\n")
    return list(links)


def scrape_recipe(url: str):
    soup = fetch_soup(url)
    if soup is None:
        return None
    title_tag = soup.find("h1")
    title = title_tag.get_text(strip=True) if title_tag else url
    items = soup.select('.ingredient-list li[id^="ingredient_"]')
    cleaned = []
    for li in items:
        span = li.find("span")
        if span is None:
            continue
        cleaned_name = clean_ingredient_name(span.get_text(" ", strip=True))
        if cleaned_name:
            cleaned.append(cleaned_name)
    if not cleaned:
        print(f"  [peringatan] Tidak ada bahan terbaca di {url}")
        return None
    return {"url": url, "title": title, "ingredients": cleaned}


def load_existing_raw():
    if os.path.exists(RAW_JSON):
        with open(RAW_JSON, "r", encoding="utf-8") as f:
            return json.load(f)
    return []


def save_raw(recipes):
    os.makedirs(OUT_DIR, exist_ok=True)
    with open(RAW_JSON, "w", encoding="utf-8") as f:
        json.dump(recipes, f, ensure_ascii=False, indent=2)


def build_network(recipes):
    node_ids = {}
    edge_weights = {}

    def get_or_create_id(name):
        if name not in node_ids:
            node_ids[name] = len(node_ids) + 1
        return node_ids[name]

    for recipe in recipes:
        ids_in_recipe = sorted({get_or_create_id(ing) for ing in recipe["ingredients"]})
        for a, b in combinations(ids_in_recipe, 2):
            edge_weights[(a, b)] = edge_weights.get((a, b), 0) + 1

    os.makedirs(OUT_DIR, exist_ok=True)

    with open(NODES_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "label"])
        for name, node_id in sorted(node_ids.items(), key=lambda x: x[1]):
            writer.writerow([node_id, name])

    with open(EDGES_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["source", "target", "weight"])
        for (a, b), w in edge_weights.items():
            writer.writerow([a, b, w])

    print(f"\nSelesai. {len(node_ids)} node (bahan unik), {len(edge_weights)} edge "
          f"(pasangan bahan) dari {len(recipes)} resep.")
    print(f"Tersimpan di: {NODES_CSV} dan {EDGES_CSV}")

    if len(node_ids) < 50:
        print("\n[PERHATIAN] Node masih di bawah 50 (syarat minimum bukti pilot "
              "proposal). Tambah PAGES_PER_KEYWORD atau SEARCH_KEYWORDS lalu run lagi.")


def main():
    check_robots_txt()
    recipes = load_existing_raw()
    print(f"Sudah ada {len(recipes)} resep tersimpan sebelumnya (checkpoint).\n")
    scraped_urls = {r["url"] for r in recipes}
    links = [l for l in collect_recipe_links() if l not in scraped_urls]

    for link in links:
        if len(recipes) >= TARGET_RECIPES_PILOT:
            print("Target pilot tercapai, berhenti.")
            break
        print(f"[{len(recipes)+1}/{TARGET_RECIPES_PILOT}] Scraping: {link}")
        data = scrape_recipe(link)
        if data:
            recipes.append(data)
        time.sleep(RATE_LIMIT_DELAY)
        if len(recipes) % 10 == 0:
            save_raw(recipes)
            print(f"  -> checkpoint tersimpan ({len(recipes)} resep)\n")

    save_raw(recipes)
    build_network(recipes)


main()

--- robots.txt cookpad.com (ringkas) ---
  User-agent: *
  Disallow: /user/confirm_premium_navi
  Disallow: /*?_pxhc=*
  Disallow: /*/recipes/*/reactions
  Disallow: /*/recipes/*/similar_recipes
  Disallow: /*/recipes/*/newer_premium_recipes
  Disallow: /*/recipes/*/cooksnaps_section
  Disallow: /*/me/recipes/*/bookmark_folders
  Disallow: /*/recipes/*/print
  Disallow: /*/activity_logs
  Disallow: /*/service_feedbacks
  Allow: /
-> Path /id/cari/ dan /id/resep/ TIDAK ada di daftar Disallow untuk User-agent: * -> aman untuk discraping script ini.

Sudah ada 0 resep tersimpan sebelumnya (checkpoint).

Ambil daftar resep: https://cookpad.com/id/cari/rendang?page=1
Ambil daftar resep: https://cookpad.com/id/cari/rendang?page=2
Ambil daftar resep: https://cookpad.com/id/cari/rendang?page=3
Total link resep unik ditemukan: 90

[1/60] Scraping: https://cookpad.com/id/resep/26568567
[2/60] Scraping: https://cookpad.com/id/resep/26541554
[3/60] Scraping: https://cookpad.com/id/resep/26647840
[

In [14]:
import shutil
shutil.make_archive("dataset_rendang", "zip", "output")

'/content/dataset_rendang.zip'